In [1]:
import sys;sys.path.append('../..')
from abslithist import *

In [12]:
df=pd.read_csv(os.path.expanduser('~/lltk_data/corpora/hathi_essays/metadata.csv'))
df['decade'] = df.year//10*10

In [13]:
norm_d = get_norm_dict()
# norm_d

In [14]:
def score_freqs(path_json):
    with open(path_json) as f:
        freqs = json.load(f)
    out = []
    for word,count in freqs.items():
        if word in norm_d:
            for n in range(count):
                out.append(norm_d[word])
    return np.mean(out)

In [15]:
score_freqs('/Users/ryan/lltk_data/corpora/hathi_essays/freqs/bc/ark:/13960/t3tt50f40.json')

np.float64(-0.24320686808933187)

In [16]:
folder = os.path.expanduser('~/lltk_data/corpora/hathi_essays/freqs/')
folder

'/Users/ryan/lltk_data/corpora/hathi_essays/freqs/'

In [17]:
def score_freqs_folder(folder, total=None):
    out = []
    iterr = tqdm(total=total)
    for root,dirs,files in os.walk(folder):
        for file in files:
            if file.endswith('.json'):
                iterr.update(1)
                path_json = os.path.join(root,file)
                score = score_freqs(path_json)
                out.append({'id':path_json.replace(folder,'').replace('.json',''), 'score':score})
    return pd.DataFrame(out)

In [ ]:
df_scores = score_freqs_folder(folder, total=len(df))
df_scores

100%|█████████▉| 7109/7142 [00:52<00:00, 134.21it/s]


,id,score
0,chi/22363535,-0.309181
1,chi/24341391,-0.359720
2,chi/41971619,-0.500596
3,chi/19044512,-0.884068
4,chi/23493467,-0.610829
...,...,...
7104,nnc1/cu56748485,-0.726034
7105,nnc1/cu01498452,-0.516506
7106,nnc1/cr59940794,-0.385489
7107,nnc1/cr60148667,-0.651433


In [25]:
odf=df_scores.dropna().merge(df, on='id', how='left').set_index('id').sort_values('score',ascending=False)
odf

,score,author,title,year,htid,access,rights,ht_bib_key,description,source,...,lang,bib_fmt,collection_code,content_provider_code,responsible_entity_code,digitization_agent_code,access_profile_code,genre,year_orig,decade
id,,,,,,,,,,,,,,,,,,,,,
mdp/39015009149728,0.893770,Aristotle.,The politics of Aristotle; trans. into English...,1885.0,mdp.39015009149728,allow,pdus,399622,v.2 pt.1,MIU,...,eng,BK,MIU,umich,umich,google,google,Essay,1885.0,1880.0
mdp/39015002174590,0.643044,"Wolfe, Bertram David, 1896-1977.",Revolution and reality : essays on the origin ...,1981.0,mdp.39015002174590,deny,ic,127682,NaN,MIU,...,eng,BK,MIU,umich,umich,google,google,Essay,1981.0,1980.0
mdp/39015068969974,0.625741,NaN,Monument in cantos and essays.,NaN,mdp.39015068969974,deny,ic,59100,no.8 1982,MIU,...,eng,SE,MIU,umich,umich,google,google,Essay,NaN,NaN
mdp/39015048693801,0.594082,"Williamson, Henry, 1895-1977.",The lone swallows : and other essays of boyhoo...,1984.0,mdp.39015048693801,deny,ic,341139,NaN,MIU,...,eng,BK,MIU,umich,umich,google,google,Essay,1984.0,1980.0
mdp/39015005387835,0.537058,NaN,Tutorial essays in psychology : a guide to rec...,1977.0,mdp.39015005387835,deny,ic,23774,v.2,MIU,...,eng,BK,MIU,umich,umich,google,google,Essay,1977.0,1970.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mdp/39015023645644,-1.023992,"Milne, A. J. M. 1922-",Human rights and human diversity : an essay in...,1986.0,mdp.39015023645644,deny,ic,492269,NaN,MIU,...,eng,BK,MIU,umich,umich,google,google,Essay,1986.0,1980.0
uc1/b4244719,-1.031961,"Raz, Joseph.",The authority of law : essays on law and moral...,1979.0,uc1.b4244719,deny,ic,303361,copy 2,UC,...,eng,BK,NRLF,universityofcalifornia,universityofcalifornia,google,google,Essay,1979.0,1970.0
mdp/39015066045918,-1.032370,"Raz, Joseph.",The authority of law : essays on law and moral...,1979.0,mdp.39015066045918,deny,ic,303361,NaN,MIU,...,eng,BK,MIU,umich,umich,google,google,Essay,1979.0,1970.0


In [26]:
odf.to_pickle('../../data/scores/v3/data.scores.HathiEssays.pkl')